#### Setup



In [1]:
%pip install -q sentence-transformers
%pip install -q wikipedia-api
%pip install -q numpy
%pip install -q scipy

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


### Load the Embedding Model:

In [2]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("Alibaba-NLP/gte-base-en-v1.5", trust_remote_code=True)

e:\1.Projects\Gen-AI-notes\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
A new version of the following files was downloaded from https://huggingface.co/Alibaba-NLP/new-impl:
- configuration.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
A new version of the following files was downloaded from https://huggingface.co/Alibaba-NLP/new-impl:
- modeling.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


### Fetch Text Content from Wikipedia:



In [3]:
from wikipediaapi import Wikipedia
wiki = Wikipedia('RAGBot/0.0', 'en')
doc = wiki.page('Hayao_Miyazaki').text
paragraphs = doc.split('\n\n') # chunking


In [4]:
import textwrap


In [5]:
for i, p in enumerate(paragraphs):
  wrapped_text = textwrap.fill(p, width=100)

  print("-----------------------------------------------------------------")
  print(wrapped_text)
  print("-----------------------------------------------------------------")


-----------------------------------------------------------------
Hayao Miyazaki (宮崎 駿 or 宮﨑 駿, Miyazaki Hayao; [mijaꜜzaki hajao]; born January 5, 1941) is a Japanese
animator, filmmaker, and manga artist. He co-founded Studio Ghibli and serves as its honorary
chairman. Throughout his career, Miyazaki has attained international acclaim as a masterful
storyteller and creator of Japanese animated feature films, and is widely regarded as one of the
most accomplished filmmakers in the history of animation. Born in Tokyo City, Miyazaki expressed
interest in manga and animation from an early age. He joined Toei Animation in 1963, working as an
inbetween artist and key animator on films like Gulliver's Travels Beyond the Moon (1965), Puss in
Boots (1969), and Animal Treasure Island (1971), before moving to A-Pro in 1971, where he co-
directed Lupin the Third Part I (1971–1972) alongside Isao Takahata. After moving to Zuiyō Eizō
(later Nippon Animation) in 1973, Miyazaki worked as an animator 

### Embed the Document:

In [6]:
docs_embed = model.encode(paragraphs, normalize_embeddings=True)

In [7]:
docs_embed.shape

(24, 768)

In [8]:
docs_embed[0]

array([ 8.84080026e-03, -3.74024780e-03,  4.59099673e-02, -1.97348539e-02,
       -1.73038919e-03, -2.20876541e-02,  4.54153642e-02,  4.27795798e-02,
        5.10134622e-02, -7.44637661e-03, -4.98125050e-03,  8.49887542e-03,
        3.33748758e-02,  1.77585315e-02, -2.42054369e-02,  1.50607694e-02,
        1.39943631e-02, -3.53626162e-02, -2.50747297e-02, -3.05661596e-02,
        5.96343540e-03,  8.10209662e-03, -2.66501177e-02,  3.49834748e-02,
       -6.55331463e-03, -4.18885946e-02, -3.31337042e-02, -1.77236982e-02,
       -2.86285989e-02, -3.93237453e-03,  2.61045378e-02, -1.18499845e-02,
        8.44645128e-03,  1.21832816e-02, -2.85338033e-02, -5.73640503e-02,
       -4.25445735e-02,  5.40513620e-02, -2.37289499e-02, -3.26739736e-02,
       -6.78446442e-02,  1.20941782e-02, -2.85129156e-02, -1.64674036e-02,
       -5.28072640e-02,  1.72322206e-02,  1.83113255e-02, -3.79651636e-02,
        4.19695023e-03, -1.44036906e-02, -4.17688265e-02,  4.37125936e-03,
       -9.20983788e-04, -

### Embed the Query:

In [9]:
query = "What was Studio Ghibli's first film?"
query_embed = model.encode(query, normalize_embeddings=True)


In [10]:
query_embed.shape

(768,)

### Find the Closest Paragraphs to the Query:



In [11]:
import numpy as np
similarities = np.dot(docs_embed, query_embed.T)

In [12]:
similarities.shape

(24,)

In [13]:
similarities

array([0.5545654 , 0.4559156 , 0.509673  , 0.48271346, 0.4975357 ,
       0.45998356, 0.48312214, 0.6124489 , 0.51500714, 0.5577719 ,
       0.5470549 , 0.5990713 , 0.5203816 , 0.44750118, 0.54033494,
       0.46270692, 0.41976064, 0.45126027, 0.4089863 , 0.462878  ,
       0.43716425, 0.43962172, 0.19419609, 0.46501288], dtype=float32)

In [14]:
top_3_idx = np.argsort(similarities, axis=0)[-3:][::-1].tolist()


In [15]:
top_3_idx

[7, 11, 9]

In [16]:
most_similar_documents = [paragraphs[idx] for idx in top_3_idx]

In [17]:
CONTEXT = ""
for i, p in enumerate(most_similar_documents):
  wrapped_text = textwrap.fill(p, width=100)

  print("-----------------------------------------------------------------")
  print(wrapped_text)
  print("-----------------------------------------------------------------")
  CONTEXT += wrapped_text + "\n\n"

-----------------------------------------------------------------
Studio Ghibli Foundation and Laputa (1985–1987) Following the success of Nausicaä of the Valley of
the Wind, Miyazaki and Takahata founded the animation production company Studio Ghibli on June 15,
1985, as a subsidiary of Tokuma Shoten, with offices in Kichijōji designed by Miyazaki. The studio's
name had been registered a year earlier; Miyazaki named it after the nickname of the Caproni Ca.309
aircraft, meaning "a hot wind that blows in the desert" in Italian. Suzuki worked for Studio Ghibli
as producer, joining full-time in 1989, while Topcraft's Tōru Hara became production manager;
Suzuki's role in the creation of the studio and its films has led him to being occasionally named a
co-founder, and Hara is often viewed as influential to the company's success. Yasuyoshi Tokuma, the
founder of Tokuma Shoten, was also closely related to the company's creation, having provided
financial backing. Topcraft had been considered

In [18]:
query = "What was Studio Ghibli's first film?"

In [19]:
prompt = f"""
use the following CONTEXT to answer the QUESTION at the end.
If you don't know the answer, just say that you don't know, don't try to make up an answer.

CONTEXT: {CONTEXT}
QUESTION: {query}

"""

In [22]:
%pip install -q openai

Note: you may need to restart the kernel to use updated packages.


In [21]:
# prompt: write python code to make calls to openai api
from google.colab import userdata
userdata.get('openai')

import openai



ModuleNotFoundError: No module named 'google.colab'

In [ ]:
from openai import OpenAI
client = OpenAI(api_key=userdata.get('openai'))

In [ ]:
response = client.chat.completions.create(
  model="gpt-4o",
  messages=[
    {"role": "user", "content": prompt},
  ]
)

In [ ]:
print(response.choices[0].message.content)
